# DTM: Topic Continuity Rate

**Scenario 3b** — Best-match assignment approach.

Each topic at t gets assigned to its **single best match** at t+1:
- **Stable**: topic maps 1-to-1 (only one t-topic points to this t+1-topic)
- **Merge**: multiple t-topics point to the same t+1-topic
- **Disappear**: best match is below threshold (topic lost)
- **New**: t+1-topic has no incoming match (emerged new)

From t-side: Stable + Merge + Disappear = 100%

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
TEMPORAL_DIR = Path("../../../../results/dtm/temporal")
RESULT_DIR = Path("../../../../results/dtm/consistency")
THRESHOLD = 0.5

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Threshold: {THRESHOLD}")
print(f"Reading from: {TEMPORAL_DIR}")
print(f"Saving to: {RESULT_DIR}")

Threshold: 0.5
Reading from: ../../../../results/dtm/temporal
Saving to: ../../../../results/dtm/consistency


In [3]:
def parse_words(words_str):
    return [w.strip() for w in str(words_str).split(",")]


def build_word_matrix(topic_ids, topic_words_dict, year, vocab_index):
    matrix = np.zeros((len(topic_ids), len(vocab_index)))
    for i, tid in enumerate(topic_ids):
        words = topic_words_dict.get((year, tid), [])
        for w in words:
            if w in vocab_index:
                matrix[i, vocab_index[w]] = 1.0
    return matrix

## Compute Continuity Rate (Best-Match)

In [4]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Continuity Rate: {subject.upper()} (DTM)")
    print(f"{'='*70}")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    topic_words = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        topic_words[key] = parse_words(row["top_words"])

    years = sorted(evo_df["year"].unique())

    all_words = set()
    for words in topic_words.values():
        all_words.update(words)
    vocab_index = {w: i for i, w in enumerate(sorted(all_words))}

    all_transition_rows = []
    all_merge_rows = []
    all_new_rows = []
    summary_rows = []

    for i in range(len(years) - 1):
        t, t_next = int(years[i]), int(years[i + 1])

        topics_t = sorted([tid for (y, tid) in topic_words if y == t])
        topics_t1 = sorted([tid for (y, tid) in topic_words if y == t_next])

        if not topics_t or not topics_t1:
            continue

        mat_t = build_word_matrix(topics_t, topic_words, t, vocab_index)
        mat_t1 = build_word_matrix(topics_t1, topic_words, t_next, vocab_index)
        sim_matrix = cosine_similarity(mat_t, mat_t1)

        # Step 1: Each topic at t gets its BEST match at t+1
        best_match_idx = np.argmax(sim_matrix, axis=1)  # index into topics_t1
        best_match_sim = np.max(sim_matrix, axis=1)

        # Step 2: Build assignment map (which t-topics point to which t1-topic)
        assignment = {}  # topic_t → best topic_t1 (or None if below threshold)
        target_counts = Counter()  # topic_t1 → how many t-topics point to it

        for idx, tid in enumerate(topics_t):
            if best_match_sim[idx] >= THRESHOLD:
                target_tid = topics_t1[best_match_idx[idx]]
                assignment[tid] = target_tid
                target_counts[target_tid] += 1
            else:
                assignment[tid] = None  # disappeared

        # Step 3: Classify each topic at t
        n_stable, n_merge, n_disappear = 0, 0, 0
        for idx, tid in enumerate(topics_t):
            target = assignment[tid]
            sim_val = float(best_match_sim[idx])
            words_t = ", ".join(topic_words.get((t, tid), []))

            if target is None:
                category = "disappear"
                n_disappear += 1
            elif target_counts[target] == 1:
                category = "stable"
                n_stable += 1
            else:
                category = "merge"
                n_merge += 1

            all_transition_rows.append({
                "subject": subject, "year_from": t, "year_to": t_next,
                "topic_id": tid, "category": category,
                "best_match_topic": target if target is not None else -1,
                "best_match_sim": round(sim_val, 6),
                "words": words_t,
            })

        # Step 4: Detect merge groups (t1-topics with >1 incoming)
        for target_tid, count in target_counts.items():
            if count > 1:
                # Find all source topics
                sources = [tid for tid, tgt in assignment.items() if tgt == target_tid]
                source_sims = []
                for src_tid in sources:
                    src_idx = topics_t.index(src_tid)
                    tgt_idx = topics_t1.index(target_tid)
                    source_sims.append(round(float(sim_matrix[src_idx, tgt_idx]), 4))

                words_t1 = ", ".join(topic_words.get((t_next, target_tid), []))
                all_merge_rows.append({
                    "subject": subject, "year_from": t, "year_to": t_next,
                    "target_topic": target_tid,
                    "n_sources": count,
                    "source_topics": str(sources),
                    "source_sims": str(source_sims),
                    "target_words": words_t1,
                })

        # Step 5: Detect new topics (t1-topics with no incoming)
        matched_t1 = set(assignment.values()) - {None}
        new_topics = [tid for tid in topics_t1 if tid not in matched_t1]
        for tid in new_topics:
            words_t1 = ", ".join(topic_words.get((t_next, tid), []))
            all_new_rows.append({
                "subject": subject, "year_from": t, "year_to": t_next,
                "topic_id": tid, "words": words_t1,
            })

        total_t = len(topics_t)
        n_new = len(new_topics)
        n_merge_groups = sum(1 for c in target_counts.values() if c > 1)

        summary_rows.append({
            "subject": subject, "year_from": t, "year_to": t_next,
            "n_topics_t": total_t, "n_topics_t1": len(topics_t1),
            "n_stable": n_stable, "n_merge": n_merge,
            "n_disappear": n_disappear, "n_merge_groups": n_merge_groups,
            "n_new": n_new,
            "pct_stable": round(n_stable / total_t * 100, 2),
            "pct_merge": round(n_merge / total_t * 100, 2),
            "pct_disappear": round(n_disappear / total_t * 100, 2),
        })

        print(f"  {t}→{t_next}: Stable={n_stable} ({n_stable/total_t:.0%})  "
              f"Merge={n_merge} ({n_merge/total_t:.0%})  "
              f"Disappear={n_disappear} ({n_disappear/total_t:.0%})  "
              f"New={n_new}")

    # Save CSVs
    pd.DataFrame(all_transition_rows).to_csv(
        RESULT_DIR / subject / "continuity_transitions.csv", index=False)
    pd.DataFrame(all_merge_rows).to_csv(
        RESULT_DIR / subject / "continuity_merges.csv", index=False)
    pd.DataFrame(all_new_rows).to_csv(
        RESULT_DIR / subject / "continuity_new_topics.csv", index=False)

    sum_df = pd.DataFrame(summary_rows)
    sum_df.to_csv(RESULT_DIR / subject / "continuity_summary.csv", index=False)

    subj_sum = sum_df[sum_df["subject"] == subject]
    overall = {
        "subject": subject, "threshold": THRESHOLD,
        "avg_pct_stable": round(subj_sum["pct_stable"].mean(), 2),
        "avg_pct_merge": round(subj_sum["pct_merge"].mean(), 2),
        "avg_pct_disappear": round(subj_sum["pct_disappear"].mean(), 2),
        "total_merge_groups": int(subj_sum["n_merge_groups"].sum()),
        "total_new": int(subj_sum["n_new"].sum()),
    }
    pd.DataFrame([overall]).to_csv(
        RESULT_DIR / subject / "continuity_overall.csv", index=False)

    print(f"\n  Overall: Stable={overall['avg_pct_stable']:.1f}%  "
          f"Merge={overall['avg_pct_merge']:.1f}%  "
          f"Disappear={overall['avg_pct_disappear']:.1f}%  "
          f"New={overall['total_new']}")
    print(f"  Saved: {RESULT_DIR / subject}")


Continuity Rate: CS (DTM)
  2000→2001: Stable=2 (4%)  Merge=43 (86%)  Disappear=5 (10%)  New=42
  2001→2002: Stable=10 (20%)  Merge=36 (72%)  Disappear=4 (8%)  New=34
  2002→2003: Stable=14 (28%)  Merge=35 (70%)  Disappear=1 (2%)  New=24
  2003→2004: Stable=8 (16%)  Merge=41 (82%)  Disappear=1 (2%)  New=32
  2004→2005: Stable=9 (18%)  Merge=40 (80%)  Disappear=1 (2%)  New=35
  2005→2006: Stable=11 (22%)  Merge=39 (78%)  Disappear=0 (0%)  New=30
  2006→2007: Stable=10 (20%)  Merge=40 (80%)  Disappear=0 (0%)  New=28
  2007→2008: Stable=8 (16%)  Merge=38 (76%)  Disappear=4 (8%)  New=31
  2008→2009: Stable=13 (26%)  Merge=36 (72%)  Disappear=1 (2%)  New=28
  2009→2010: Stable=9 (18%)  Merge=37 (74%)  Disappear=4 (8%)  New=31
  2010→2011: Stable=12 (24%)  Merge=32 (64%)  Disappear=6 (12%)  New=26
  2011→2012: Stable=16 (32%)  Merge=32 (64%)  Disappear=2 (4%)  New=26
  2012→2013: Stable=12 (24%)  Merge=32 (64%)  Disappear=6 (12%)  New=28
  2013→2014: Stable=9 (18%)  Merge=32 (64%)  Disappea

## Analysis Review

In [5]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Analysis Review: {subject.upper()} (DTM)")
    print(f"{'='*70}")

    trans_df = pd.read_csv(RESULT_DIR / subject / "continuity_transitions.csv")
    merge_path = RESULT_DIR / subject / "continuity_merges.csv"
    merge_df = pd.read_csv(merge_path) if merge_path.stat().st_size > 10 else pd.DataFrame()
    new_path = RESULT_DIR / subject / "continuity_new_topics.csv"
    new_df = pd.read_csv(new_path) if new_path.stat().st_size > 10 else pd.DataFrame()
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    # --- DISAPPEARED ---
    disappeared = trans_df[trans_df["category"] == "disappear"]
    print(f"\n  📉 DISAPPEARED: {len(disappeared)} events")
    if len(disappeared) > 0:
        per_year = disappeared.groupby("year_from").size()
        print(f"  Per year:")
        for yr, cnt in per_year.items():
            print(f"    {int(yr)}: {cnt} topics")

        # Permanently gone
        perm = []
        for _, r in disappeared.iterrows():
            tid = int(r['topic_id'])
            later = evo_df[(evo_df['topic_id'] == tid) & (evo_df['year'] > int(r['year_to']))]
            if len(later) == 0:
                w = str(r['words'])[:50]
                perm.append(f"    T{tid:>3} last @ {int(r['year_from'])} (sim={r['best_match_sim']:.3f}) | {w}")
        print(f"\n  Permanently gone: {len(perm)}")
        for line in perm[:10]:
            print(line)
        if len(perm) > 10:
            print(f"    ... +{len(perm)-10} more")

    # --- MERGE GROUPS ---
    print(f"\n  🔗 MERGE GROUPS: {len(merge_df)} events")
    if len(merge_df) > 0:
        print(f"  Top 10 merges (most sources):")
        for _, r in merge_df.nlargest(10, 'n_sources').iterrows():
            w = str(r['target_words'])[:40]
            print(f"    T{int(r['target_topic']):>3} @ {int(r['year_to'])} "
                  f"← {r['source_topics']} (sims={r['source_sims']}) | {w}")

    # --- NEW TOPICS ---
    print(f"\n  🆕 NEW TOPICS: {len(new_df)} events")
    if len(new_df) > 0:
        per_year = new_df.groupby("year_to").size()
        print(f"  Per year:")
        for yr, cnt in per_year.items():
            print(f"    {int(yr)}: {cnt} new topics")

    # --- STABILITY RANKING ---
    print(f"\n  🏆 STABILITY RANKING:")
    ts = trans_df.groupby("topic_id")["category"].value_counts().unstack(fill_value=0)
    for cat in ["stable", "merge", "disappear"]:
        if cat not in ts.columns:
            ts[cat] = 0
    ts["total"] = ts.sum(axis=1)
    ts["stability_pct"] = (ts["stable"] / ts["total"] * 100).round(1)
    ts = ts.sort_values("stability_pct", ascending=False)

    print(f"  Top 5 most stable:")
    for tid, r in ts.head(5).iterrows():
        print(f"    T{int(tid):>3} | {r['stability_pct']:.0f}% stable "
              f"({int(r['stable'])}S {int(r['merge'])}M {int(r['disappear'])}D)")
    print(f"  Top 5 most unstable:")
    for tid, r in ts.tail(5).iterrows():
        print(f"    T{int(tid):>3} | {r['stability_pct']:.0f}% stable "
              f"({int(r['stable'])}S {int(r['merge'])}M {int(r['disappear'])}D)")
    print()


Analysis Review: CS (DTM)

  📉 DISAPPEARED: 147 events
  Per year:
    2000: 5 topics
    2001: 4 topics
    2002: 1 topics
    2003: 1 topics
    2004: 1 topics
    2007: 4 topics
    2008: 1 topics
    2009: 4 topics
    2010: 6 topics
    2011: 2 topics
    2012: 6 topics
    2013: 9 topics
    2014: 14 topics
    2015: 27 topics
    2016: 32 topics
    2017: 20 topics
    2018: 8 topics
    2019: 1 topics
    2020: 1 topics

  Permanently gone: 0

  🔗 MERGE GROUPS: 149 events
  Top 10 merges (most sources):
    T  1 @ 2001 ← [0, 2, 3, 4, 6, 13, 14, 15, 16, 22, 23, 24, 25, 28, 34, 37, 42, 43, 44, 49] (sims=[0.5, 0.8, 0.5, 0.6, 0.7, 0.5, 0.5, 0.6, 0.7, 0.6, 0.5, 0.6, 0.5, 0.7, 0.7, 0.7, 0.5, 0.8, 0.6, 0.6]) | algorithm, program, constraint, informat
    T  2 @ 2005 ← [0, 1, 4, 10, 11, 14, 16, 17, 19, 20, 24, 25, 27, 29, 31, 33, 37, 38, 41, 46] (sims=[0.6, 0.6, 0.7, 0.6, 0.7, 0.7, 0.6, 0.6, 0.6, 0.6, 0.7, 0.7, 0.7, 0.6, 0.7, 0.6, 0.6, 0.5, 0.6, 0.7]) | datum, network, information, co